
# Caderno Único de Trading — LULU, GAP (GPS), ANF (Yahoo Finance)
**FIX-4** — fallback automático do Yahoo Finance + tratamento de erros por ticker.
Inclui gráficos 3M/1M/7D/1D, 24h (1h), MACD/RSI, EMA21, EMA50/EMA200 (Golden/Death),
MACD filtrado, ADX/ATR, sizing por ATR, alarmes (e-mail/webhook) e sanitização robusta de OHLC.
*Gerado em:* **2025-10-05**


## 0) Dependências

In [ ]:

import sys, subprocess
def _pip_install(pkg):
    try:
        __import__(pkg.split("==")[0])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])
for p in ["yfinance>=0.2.40","pandas>=2.0.0","matplotlib>=3.7.0","mplfinance>=0.12.10b0","pytz","numpy>=1.24.0","requests>=2.31.0"]:
    _pip_install(p)
print("Dependências OK.")


## 1) Configurações

In [ ]:

import os
TICKERS_INPUT = ["LULU","GAP","ANF"]
TICKER_MAP = {"GAP":"GPS"}
LOCAL_TZ = "America/Sao_Paulo"
EQUITY = 100000.0; RISCO_PCT = 0.01; ATR_MULT = 2.0
INTERVALO_POR_PERIODO = {"3mo":"1d","1mo":"1d","7d":"60m","1d":"30m"}

EMAIL_ENABLED=False; WEBHOOK_ENABLED=False
SMTP_SERVER=os.getenv("SMTP_SERVER","smtp.seuservidor.com"); SMTP_PORT=int(os.getenv("SMTP_PORT","587"))
SMTP_USER=os.getenv("SMTP_USER","usuario"); SMTP_PASSWORD=os.getenv("SMTP_PASSWORD","senha")
SMTP_USE_TLS=True; SMTP_FROM=os.getenv("SMTP_FROM","alertas@seudominio.com"); EMAIL_TO=os.getenv("EMAIL_TO","voce@seudominio.com")
WEBHOOK_URL=os.getenv("WEBHOOK_URL","https://exemplo.com/webhook"); WEBHOOK_HEADERS={"Content-Type":"application/json"}
ALERT_LOG_PATH="alert_log.csv"


## 2) Utilitários + Sanitização robusta + Fallback do Yahoo

In [ ]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt, yfinance as yf, mplfinance as mpf
import pytz, json, csv, hashlib, smtplib, requests
from email.message import EmailMessage
from datetime import datetime, timedelta
from pathlib import Path

def resolve_ticker(t): return TICKER_MAP.get(t,t)

def _download_once(ticker, period, interval):
    return yf.download(ticker, period=period, interval=interval, auto_adjust=False, progress=False)

def baixar_dados(ticker, period, interval, try_fallback=True):
    order_periods = [period, "6mo", "1y", "2y", "5y", "max"]
    order_intervals = [interval] if not any(k in interval for k in ["m","h"]) else ["30m","60m","90m","1h"]
    last_exc = None
    for per in order_periods:
        for itv in order_intervals:
            try:
                df = _download_once(ticker, per, itv)
                if isinstance(df, pd.DataFrame) and not df.empty:
                    df = df.dropna().copy()
                    if not getattr(df.index,"tz",None):
                        df.index = df.index.tz_localize("UTC")
                    return df
            except Exception as e:
                last_exc = e
        if not try_fallback: break
    print(f"[WARN] Falha ao baixar {ticker}. Último erro: {last_exc}")
    return None

def fuso_horario(df, tz):
    if df is None or df.empty: return df
    return df.tz_convert(tz)

def _flatten_ohlc_multiindex(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df.columns, pd.MultiIndex): return df
    try:
        lv0 = df.columns.get_level_values(0)
        if {"Open","High","Low","Close"}.issubset(set(lv0)):
            first = df.columns.get_level_values(1).unique()[0]
            return df.xs(first, level=1, axis=1)
        lv1 = df.columns.get_level_values(1)
        if {"Open","High","Low","Close"}.issubset(set(lv1)):
            first = df.columns.get_level_values(0).unique()[0]
            return df.xs(first, level=0, axis=1)
    except Exception: pass
    flat = df.copy(); flat.columns = ["_".join(map(str,c)).strip() for c in flat.columns.to_list()]
    out = pd.DataFrame(index=flat.index)
    for base in ["Open","High","Low","Close","Adj Close","Volume"]:
        cand = [c for c in flat.columns if c.split("_")[0].lower()==base.lower()]
        if cand: out[base] = flat[cand[0]]
    return out if not out.empty else flat

def sanitize_ohlc(df: pd.DataFrame, drop_tz_for_mpl=False) -> pd.DataFrame:
    if df is None or df.empty: return df
    out = _flatten_ohlc_multiindex(df).copy()
    out.columns = [str(c).title() for c in out.columns]
    out = out.loc[:, ~out.columns.duplicated()].copy()
    def _coerce(obj, index):
        if isinstance(obj, pd.DataFrame):
            tmp = obj.apply(lambda s: pd.to_numeric(s, errors="coerce"))
            return tmp.bfill(axis=1).iloc[:,0]
        if isinstance(obj, pd.Series):
            return pd.to_numeric(obj, errors="coerce")
        return pd.to_numeric(pd.Series(obj, index=index), errors="coerce")
    for c in ["Open","High","Low","Close","Adj Close","Volume"]:
        if c in out.columns:
            out[c] = _coerce(out[c], out.index)
    needed = [c for c in ["Open","High","Low","Close"] if c in out.columns]
    out = out.dropna(subset=needed)
    if drop_tz_for_mpl and getattr(out.index,"tz",None) is not None:
        out.index = out.index.tz_localize(None)
    return out

def ema(s, n): return s.ewm(span=n, adjust=False).mean()
def rsi(series, period=14):
    d=series.diff(); up=np.where(d>0,d,0.0); dn=np.where(d<0,-d,0.0)
    ru=pd.Series(up,index=series.index).ewm(alpha=1/period,adjust=False).mean()
    rd=pd.Series(dn,index=series.index).ewm(alpha=1/period,adjust=False).mean()
    rs=ru/rd.replace(0,np.nan); r=100-(100/(1+rs)); return r.fillna(50.0)
def macd(series, fast=12, slow=26, signal=9):
    ef,es=ema(series,fast), ema(series,slow); m=ef-es; s=m.ewm(span=signal,adjust=False).mean(); return m,s,m-s
def true_range(h,l,c): pc=c.shift(1); return pd.concat([h-l,(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
def atr(h,l,c,period=14): return true_range(h,l,c).ewm(alpha=1/period,adjust=False).mean()
def adx(h,l,c,period=14):
    up=h.diff(); dn=-l.diff()
    plus_dm=np.where((up>dn)&(up>0),up,0.0); minus_dm=np.where((dn>up)&(dn>0),dn,0.0)
    tr=true_range(h,l,c); a=tr.ewm(alpha=1/period,adjust=False).mean()
    plus_di=100*(pd.Series(plus_dm,index=h.index).ewm(alpha=1/period,adjust=False).mean()/a)
    minus_di=100*(pd.Series(minus_dm,index=h.index).ewm(alpha=1/period,adjust=False).mean()/a)
    dx=(100*(plus_di-minus_di).abs()/(plus_di+minus_di).replace(0,np.nan)).fillna(0)
    return dx.ewm(alpha=1/period,adjust=False).mean()
def crossover_up(a,b): return (a.shift(1)<=b.shift(1)) & (a>b)
def crossover_down(a,b): return (a.shift(1)>=b.shift(1)) & (a<b)


## 3) Plotters

In [ ]:

def plot_preco_simples(df,titulo=""):
    df=sanitize_ohlc(df,False); 
    if df is None or df.empty: print("[WARN] Sem dados para", titulo); return
    plt.figure(figsize=(10,4)); plt.plot(df.index,df["Close"],label="Close")
    plt.title(titulo or "Preço (Close)"); plt.xlabel("Tempo"); plt.ylabel("Preço"); plt.grid(True); plt.legend(); plt.show()

def plot_candles(df,titulo=""):
    df=sanitize_ohlc(df,True)
    if df is None or df.empty or len(df)<3: print("Sem candles suficientes para plotar."); return
    mpf.plot(df,type="candle",style="yahoo",title=titulo,volume=False)

def plot_macd(df,titulo=""):
    df=sanitize_ohlc(df,False)
    if df is None or df.empty: print("[WARN] Sem dados para", titulo); return
    close=pd.to_numeric(df["Close"],errors="coerce").dropna()
    m,s,h=macd(close); plt.figure(figsize=(10,4)); plt.plot(m.index,m,label="MACD"); plt.plot(s.index,s,label="Signal")
    buys=crossover_up(m,s); sells=crossover_down(m,s)
    plt.scatter(m.index[buys],m[buys],marker="^",label="Buy"); plt.scatter(m.index[sells],m[sells],marker="v",label="Sell")
    plt.title(titulo or "MACD"); plt.xlabel("Tempo"); plt.ylabel("Valor"); plt.grid(True); plt.legend(); plt.show()

def plot_macd_filtrado(df,ema50,ema200,titulo=""):
    df=sanitize_ohlc(df,False)
    if df is None or df.empty: print("[WARN] Sem dados para", titulo); return
    close=pd.to_numeric(df["Close"],errors="coerce").dropna()
    ema50=ema50.reindex(close.index).ffill(); ema200=ema200.reindex(close.index).ffill()
    m,s,h=macd(close); bull=(close>ema50)&(ema50>ema200); bear=(close<ema50)&(ema50<ema200)
    buys_raw=crossover_up(m,s); sells_raw=crossover_down(m,s)
    buys,sells=buys_raw & bull, sells_raw & bear
    plt.figure(figsize=(10,4)); plt.plot(m.index,m,label="MACD"); plt.plot(s.index,s,label="Signal")
    plt.scatter(m.index[buys],m[buys],marker="^",label="Buy (filtro EMA50/200)")
    plt.scatter(m.index[sells],m[sells],marker="v",label="Sell (filtro EMA50/200)")
    plt.title(titulo or "MACD (filtrado por EMA50/200)"); plt.xlabel("Tempo"); plt.ylabel("Valor"); plt.grid(True); plt.legend(); plt.show()

def plot_rsi(df,titulo=""):
    df=sanitize_ohlc(df,False)
    if df is None or df.empty: print("[WARN] Sem dados para", titulo); return
    r=rsi(df["Close"]); plt.figure(figsize=(10,3)); plt.plot(r.index,r,label="RSI(14)")
    plt.axhline(70,linestyle="--"); plt.axhline(30,linestyle="--")
    ob=(r.shift(1)<70)&(r>=70); os=(r.shift(1)>30)&(r<=30)
    plt.scatter(r.index[ob],r[ob],marker="x",label="Overbought"); plt.scatter(r.index[os],r[os],marker="x",label="Oversold")
    plt.title(titulo or "RSI(14)"); plt.xlabel("Tempo"); plt.ylabel("RSI"); plt.grid(True); plt.legend(); plt.show()

def plot_ema21_com_sinais(df,titulo=""):
    df=sanitize_ohlc(df,False)
    if df is None or df.empty: print("[WARN] Sem dados para", titulo); return
    px=df.copy(); px["EMA21"]=ema(px["Close"],21)
    ent=crossover_up(px["Close"],px["EMA21"]); sai=crossover_down(px["Close"],px["EMA21"])
    plt.figure(figsize=(11,5)); plt.plot(px.index,px["Close"],label="Close"); plt.plot(px.index,px["EMA21"],label="EMA21")
    plt.scatter(px.index[ent],px["Close"][ent],marker="^",label="Entrada"); plt.scatter(px.index[sai],px["Close"][sai],marker="v",label="Saída")
    plt.title(titulo or "EMA21 - Entradas/Saídas"); plt.xlabel("Tempo"); plt.ylabel("Preço"); plt.grid(True); plt.legend(); plt.show()

def plot_ema50200(df,titulo=""):
    df=sanitize_ohlc(df,False)
    if df is None or df.empty: print("[WARN] Sem dados para", titulo); return pd.Series(dtype=float),pd.Series(dtype=float),None,None
    px=df.copy(); px["EMA50"]=ema(px["Close"],50); px["EMA200"]=ema(px["Close"],200)
    golden=crossover_up(px["EMA50"],px["EMA200"]); death=crossover_down(px["EMA50"],px["EMA200"])
    plt.figure(figsize=(11,5)); plt.plot(px.index,px["Close"],label="Close"); plt.plot(px.index,px["EMA50"],label="EMA50"); plt.plot(px.index,px["EMA200"],label="EMA200")
    plt.scatter(px.index[golden],px["EMA50"][golden],marker="^",label="Golden Cross (50>200)")
    plt.scatter(px.index[death],px["EMA50"][death],marker="v",label="Death Cross (50<200)")
    plt.title(titulo or "EMA50/EMA200 — Regime e Crossovers"); plt.xlabel("Tempo"); plt.ylabel("Preço"); plt.grid(True); plt.legend(); plt.show()
    return px["EMA50"],px["EMA200"],golden,death

def plot_adx(df,titulo=""):
    df=sanitize_ohlc(df,False)
    if df is None or df.empty: print("[WARN] Sem dados para", titulo); return pd.Series(dtype=float)
    a=adx(df["High"],df["Low"],df["Close"],14)
    plt.figure(figsize=(10,3)); plt.plot(a.index,a,label="ADX(14)"); plt.axhline(25,linestyle="--")
    plt.title(titulo or "ADX(14) — Força da Tendência"); plt.xlabel("Tempo"); plt.ylabel("ADX"); plt.grid(True); plt.legend(); plt.show()
    return a

def plot_atr(df,titulo=""):
    df=sanitize_ohlc(df,False)
    if df is None or df.empty: print("[WARN] Sem dados para", titulo); return pd.Series(dtype=float)
    a=atr(df["High"],df["Low"],df["Close"],14)
    plt.figure(figsize=(10,3)); plt.plot(a.index,a,label="ATR(14)"); plt.title(titulo or "ATR(14) — Volatilidade")
    plt.xlabel("Tempo"); plt.ylabel("ATR"); plt.grid(True); plt.legend(); plt.show(); return a


## 4) Alarmes, e-mail/webhook e sizing por ATR

In [ ]:

def _alert_key(t,e,qu): return hashlib.sha256(f"{t}|{e}|{qu}".encode("utf-8")).hexdigest()
def _is_dup(k,p):
    path=Path(p)
    if not path.exists(): return False
    with path.open("r",newline="",encoding="utf-8") as f:
        for row in csv.reader(f):
            if row and row[0]==k: return True
    return False
def _append(k,p):
    with Path(p).open("a",newline="",encoding="utf-8") as f: csv.writer(f).writerow([k,datetime.utcnow().isoformat()])
def enviar_email(assunto,corpo):
    if not EMAIL_ENABLED: print("[EMAIL desabilitado] Preview:",assunto); print(corpo); return
    msg=EmailMessage(); msg["Subject"]=assunto; msg["From"]=SMTP_FROM; msg["To"]=EMAIL_TO; msg.set_content(corpo)
    with smtplib.SMTP(SMTP_SERVER,SMTP_PORT) as s:
        if SMTP_USE_TLS: s.starttls()
        if SMTP_USER: s.login(SMTP_USER,SMTP_PASSWORD)
        s.send_message(msg)
def enviar_webhook(payload):
    if not WEBHOOK_ENABLED: print("[WEBHOOK desabilitado] Preview payload:",json.dumps(payload,ensure_ascii=False)); return
    r=requests.post(WEBHOOK_URL,headers=WEBHOOK_HEADERS,data=json.dumps(payload)); print("Webhook status:",r.status_code, r.text[:200])
def notificar_alerta(row,extra):
    k=_alert_key(row["ticker"],row["evento"],row["quando"])
    if _is_dup(k,ALERT_LOG_PATH): print("Alerta duplicado ignorado:",row); return
    enviar_email(f"[ALERTA] {row['ticker']} — {row['evento']}",
f"Ticker: {row['ticker']}\nEvento: {row['evento']}\nQuando: {row['quando']}\nMotivo: {row['motivo']}\n\nResumo:\n- Regime: {extra.get('regime')}\n- ADX(14): {extra.get('adx')}\n- ATR(14): {extra.get('atr')}\n- Preço: {extra.get('preco')}")
    enviar_webhook({"type":"alerta_trading","ticker":row["ticker"],"evento":row["evento"],"quando":str(row["quando"]),"motivo":row["motivo"],"resumo":extra})
    _append(k,ALERT_LOG_PATH)

def regime_status(close,ema50,ema200):
    if len(close)==0 or len(ema50)==0 or len(ema200)==0: return "Indisponível"
    if close.iloc[-1]>ema50.iloc[-1]>ema200.iloc[-1]: return "Bullish (Close>EMA50>EMA200)"
    if close.iloc[-1]<ema50.iloc[-1]<ema200.iloc[-1]: return "Bearish (Close<EMA50<EMA200)"
    return "Neutro/Transição"

def gerar_alertas(ticker, df3m, ema50, ema200, adx3m, atr3m):
    alerts=[]; 
    if not (isinstance(df3m,pd.DataFrame) and not df3m.empty): return pd.DataFrame(alerts)
    if isinstance(ema50,pd.Series) and isinstance(ema200,pd.Series) and len(ema50)>0 and len(ema200)>0:
        golden=crossover_up(ema50,ema200); death=crossover_down(ema50,ema200)
        if len(golden)>0 and golden.iloc[-1]: alerts.append({"ticker":ticker,"evento":"Golden Cross","quando":df3m.index[-1],"motivo":"EMA50 cruzou acima da EMA200 (regime bullish)."})
        if len(death)>0 and death.iloc[-1]:   alerts.append({"ticker":ticker,"evento":"Death Cross","quando":df3m.index[-1],"motivo":"EMA50 cruzou abaixo da EMA200 (regime bearish)."})
    if isinstance(adx3m,pd.Series) and len(adx3m)>2 and (adx3m.iloc[-1]>25) and (adx3m.iloc[-1]>adx3m.iloc[-2]):
        alerts.append({"ticker":ticker,"evento":"ADX Forte/Subindo","quando":df3m.index[-1],"motivo":"Força da tendência aumentando (ADX>25)."})
    if isinstance(atr3m,pd.Series) and len(atr3m)>2 and (atr3m.iloc[-1]>atr3m.iloc[-2]>atr3m.iloc[-3]):
        alerts.append({"ticker":ticker,"evento":"ATR Crescente","quando":df3m.index[-1],"motivo":"Volatilidade em alta; ampliar stop e reduzir tamanho da posição."})
    return pd.DataFrame(alerts)


## 5) Pipeline por ticker

In [ ]:

def analisar_ticker(ticker_in):
    disp=ticker_in; tk=resolve_ticker(ticker_in)
    print(f"\n=== {disp} (Yahoo: {tk}) ===")
    d3m=baixar_dados(tk,"3mo",INTERVALO_POR_PERIODO["3mo"]); d1m=baixar_dados(tk,"1mo",INTERVALO_POR_PERIODO["1mo"])
    d7d=baixar_dados(tk,"7d",INTERVALO_POR_PERIODO["7d"]);   d1d=baixar_dados(tk,"1d",INTERVALO_POR_PERIODO["1d"])
    d2d_1h=baixar_dados(tk,"2d","60m")
    d2d_1h_z=fuso_horario(d2d_1h,LOCAL_TZ) if d2d_1h is not None else None
    ult_24h=None
    if d2d_1h_z is not None and not d2d_1h_z.empty:
        corte=d2d_1h_z.index.max()-pd.Timedelta(hours=24); ult_24h=d2d_1h_z.loc[d2d_1h_z.index>corte].copy()
    d7d_z=fuso_horario(d7d,LOCAL_TZ) if d7d is not None else None
    d1d_z=fuso_horario(d1d,LOCAL_TZ) if d1d is not None else None

    if d3m is not None: plot_preco_simples(d3m, f"{disp} — Close (últimos 3 meses)")
    else: print("[WARN] Sem dados 3M para", disp)

    if ult_24h is None or ult_24h.empty or len(ult_24h)<3: print("Poucos/nenhum candle nas últimas 24h; pulando candles 1h.")
    else: plot_candles(ult_24h, f"{disp} — Candles 1h (últimas 24h) [TZ: {LOCAL_TZ}]")

    for df,label in [(d3m,"3M, 1d"),(d1m,"1M, 1d"),(d7d_z,"7D, 60m"),(d1d_z,"1D, 30m")]:
        if df is not None: plot_macd(df, f"{disp} — MACD ({label})")

    ema50_3m=ema200_3m=None
    if d3m is not None:
        ema50_3m,ema200_3m,_,_=plot_ema50200(d3m, f"{disp} — EMA50/EMA200 (3M, 1d)")
        plot_macd_filtrado(d3m,ema50_3m,ema200_3m,f"{disp} — MACD (3M) filtrado por EMA50/200")
        plot_rsi(d3m, f"{disp} — RSI(14) (3M, 1d)")
    if d1m is not None:  plot_rsi(d1m,  f"{disp} — RSI(14) (1M, 1d)")
    if d7d_z is not None: plot_rsi(d7d_z, f"{disp} — RSI(14) (7D, 60m)")
    if d1d_z is not None: plot_rsi(d1d_z, f"{disp} — RSI(14) (1D, 30m)")

    if d3m is not None:  plot_ema21_com_sinais(d3m, f"{disp} — EMA21 Entradas/Saídas (3M, 1d)")

    adx3m = plot_adx(d3m, f"{disp} — ADX(14) (3M, 1d)") if d3m is not None else pd.Series(dtype=float)
    atr3m = plot_atr(d3m, f"{disp} — ATR(14) (3M, 1d)") if d3m is not None else pd.Series(dtype=float)

    if d3m is not None and ema50_3m is not None and ema200_3m is not None:
        close3=sanitize_ohlc(d3m)["Close"]; m3,s3,_=macd(close3)
        buys_raw=crossover_up(m3,s3); sells_raw=crossover_down(m3,s3)
        bull=(close3>ema50_3m)&(ema50_3m>ema200_3m); bear=(close3<ema50_3m)&(ema50_3m<ema200_3m)
        buys_f, sells_f = buys_raw & bull, sells_raw & bear
        def last_idx(m): return m.index[m].max() if m.any() else None
        last_buy_raw,last_sell_raw=last_idx(buys_raw),last_idx(sells_raw)
        last_buy_f,last_sell_f=last_idx(buys_f),last_idx(sells_f)

        r=rsi(close3); last_r=r.iloc[-1]
        rtxt=("Overbought" if last_r>=70 else ("Oversold" if last_r<=30 else ("Tendência forte" if last_r>=60 else ("Tendência fraca" if last_r<=40 else "Neutro"))))+f" (RSI={last_r:.1f})"
        regime_txt=regime_status(close3, ema50_3m, ema200_3m)
        resumo=pd.DataFrame([("Regime (3M)",regime_txt),("MACD 3M (bruto)",f"BUY @ {last_buy_raw}" if last_buy_raw else "—"),
                             ("MACD 3M (bruto)",f"SELL @ {last_sell_raw}" if last_sell_raw else "—"),
                             ("MACD 3M (filtrado)",f"BUY @ {last_buy_f}" if last_buy_f else "—"),
                             ("MACD 3M (filtrado)",f"SELL @ {last_sell_f}" if last_sell_f else "—"),
                             ("RSI 3M", rtxt),
                             ("ADX 3M", f"{adx3m.iloc[-1]:.1f}" if len(adx3m)>0 else "—"),
                             ("ATR 3M", f"{atr3m.iloc[-1]:.2f}" if len(atr3m)>0 else "—")], columns=["Métrica","Status"])
        display(resumo)

        alerts_df=gerar_alertas(disp, sanitize_ohlc(d3m), ema50_3m, ema200_3m, adx3m, atr3m)
        if alerts_df.empty: print("Sem alarmes no candle mais recente.")
        else:
            print("ALARMES:"); display(alerts_df)
            extra={"regime":regime_txt,"adx":float(adx3m.iloc[-1]) if len(adx3m)>0 else None,
                   "atr":float(atr3m.iloc[-1]) if len(atr3m)>0 else None,
                   "preco":float(close3.iloc[-1])}
            for _,row in alerts_df.iterrows(): notificar_alerta(row,extra)

        if len(atr3m)>0:
            preco=float(close3.iloc[-1]); av=float(atr3m.iloc[-1])
            risco_valor=EQUITY*RISCO_PCT; stop=ATR_MULT*av; qty=(risco_valor/stop) if stop>0 else 0.0
            sizing=pd.DataFrame([{"Ticker":disp,"Preço Atual":round(preco,2),"ATR(14)":round(av,3),"ATR Mult Stop":ATR_MULT,
                                  "Distância Stop":round(stop,3),"Risco por Trade":round(risco_valor,2),"Qtd. sugerida (ação aprox.)":int(qty)}])
            print("Sizing por ATR (parâmetros atuais):"); display(sizing)
        else:
            print("[WARN] ATR indisponível — pulando sizing por ATR.")
    else:
        print("[WARN] Base 3M indisponível — pulando painel/alertas/sizing.")


## 6) Rodar

In [ ]:

for tk in TICKERS_INPUT:
    try:
        analisar_ticker(tk)
    except Exception as e:
        print(f"[ERRO] Falha ao analisar {tk}: {type(e).__name__}: {e}")
